#### Imports

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime
import uuid

#### Configuration

In [0]:
#Configuration 

pipeline_name = "nyc_taxi_lakehouse_v2"
task_name = "Bronze_ingestion"

start_time = datetime.now()

run_id = str(uuid.uuid4())

print(f"Pipeline Run ID: {run_id}")
print(f"Pipeline Start Time: {start_time}")

checkpoint_location = (
    "s3://ashish-nyc-taxi-lakehouse/checkpoints/bronze"
)

schema_location = (
    "s3://ashish-nyc-taxi-lakehouse/schema/bronze"
)

Pipeline Run ID: 5bb4198f-1bba-4e3a-b777-52d01de04bb5
Pipeline Start Time: 2026-07-29 15:21:04.957163


#### Configuring Catalog , schema and adding input path

In [0]:
catalog_name = "workspace"
schema_name = "nyc_taxi_aws"

bronze_table_aws = f"{catalog_name}.{schema_name}.bronze_taxi_trips"

input_path = "s3://ashish-nyc-taxi-lakehouse/raw/"

monitoring_table = (
f"{catalog_name}.{schema_name}.pipeline_run_metrics"
)

#### Reading Raw data

In [0]:
raw_df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "parquet")
        .option("cloudFiles.useNotifications", "false")
        .option("cloudFiles.schemaLocation", schema_location)
        .load(input_path)
)

### Adding ingestion timestamp, source file and Batch id column to identify the ingested data

In [0]:
bronze_df = (
    raw_df
    .withColumn(
        "_ingestion_timestamp",
        current_timestamp()
    )
    .withColumn(
        "_source_file",
        col("_metadata.file_path")
    )
    .withColumn(
        "_batch_id",
        expr("uuid()")
    )
)


#### Creating Bronze Delta table

In [0]:
(
    bronze_df.writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        checkpoint_location
    )
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(bronze_table_aws)
)

#### updating processed data displaying updated record count in Bronze table along with the pipeline duration

In [0]:
records_processed = spark.table(
bronze_table_aws
).count()

end_time = datetime.now()

duration_seconds = (
end_time - start_time
).total_seconds()

print(
f"Bronze record count: {records_processed}"
)

print(
f"Pipeline duration: {duration_seconds:.2f} seconds"
)

Bronze record count: 12861158
Pipeline duration: 6.67 seconds
